## 0. Repository map and end-to-end flow

`generate_data.py` → `data/raw/*.csv` → audit → clean → `data/clean/*.csv` + `data/instamart.db` → RFM/segments + lapse model → privacy-safe ticket triage → policy retrieval + refund tools.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / "generate_data.py").is_file() else ROOT.parent

sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Root files:", sorted(f.name for f in ROOT.iterdir() if f.is_file()))

Project root: c:\Users\Preetha\Downloads\Swiggy-Instamart-Copilot-main\Swiggy-Instamart-Copilot-main - Karthik\instamart_customer_copilot_teaching_kit
Root files: ['.gitignore', 'GITHUB_RESOURCES.md', 'README.md', 'check_db.py', 'check_scenario_dates.py', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'run_project.py', 'test_part_d.py']


## 1. Generate / fetch the raw data

The data generator is deterministic (fixed random seed). It writes customer, product, order, support-ticket, evaluation-question, agent-scenario inputs and policy text files.


In [2]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


## 2. Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [3]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

customers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0


products: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0


tickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


### Audit questions for students
- Which columns are supposed to be unique keys?
- Are duplicates exact duplicates or multiple states of the same order?
- Which values are impossible (negative delivery duration, non-numeric money, implausible age)?
- Why should we distinguish an empty string from a missing value?
- What would go wrong if we modelled before cleaning?

In [4]:
from src.data_pipeline import audit, clean_data, validate
audit()


customers: shape=(1200, 5)
customer_id    str
city           str
signup_date    str
age            str
gender         str
dtype: object
             missing_count  missing_pct
customer_id              0          0.0
city                     0          0.0
signup_date              0          0.0
age                      0          0.0
gender                   0          0.0

products: shape=(40, 4)
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

orders: shape=(13084, 9)
order_id              str
customer_id           str
product_id            str
quantity            int64
order_ts              str
status                str
discount_pct        int64
order_amount          str
delivery_minutes      str
dtype: object
                 

## 3. Clean the data and write trusted outputs

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.

Ask students to explain why cancelled orders should keep a missing delivery duration.

In [5]:
clean = clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()

customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female


products (40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


orders (12309, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
827,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
4294,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
9973,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23


Reference after cleaning:
Cities: {'Pune': 227, 'Delhi': 210, 'Bengaluru': 197, 'Chennai': 191, 'Hyderabad': 191, 'Mumbai': 184}
Genders: {'female': 610, 'male': 560, 'unknown': 30}
Missing ages: 0
Age range: 18 to 60
ALL CHECKPOINTS PASSED


## 4. Query the SQLite database

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

Run the supplied SQL examples and compare SQL totals to pandas results.

In [16]:
import sqlite3

db_path = ROOT / "data" / "instamart.db"

with sqlite3.connect(db_path) as con:

    # 1. Display available database tables
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        con
    )
    display(tables)

    # 2. Revenue by city and product category
    q2 = pd.read_sql_query("""
        SELECT
            c.city,
            p.category,
            ROUND(SUM(o.order_amount), 2) AS revenue
        FROM orders o
        JOIN customers c USING(customer_id)
        JOIN products p USING(product_id)
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        ORDER BY c.city, p.category
    """, con)

    display(q2)

    # 3. Order summary
    q3 = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_orders,
        SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_orders,
        SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
        SUM(CASE WHEN status = 'pending' THEN 1 ELSE 0 END) AS pending_orders,
        SUM(CASE WHEN order_amount IS NULL THEN 1 ELSE 0 END) AS missing_order_amounts,
        SUM(CASE WHEN delivery_minutes IS NULL AND status = 'completed' THEN 1 ELSE 0 END) AS missing_delivery_times_completed_orders,
        SUM(CASE WHEN delivery_minutes IS NULL AND status = 'cancelled' THEN 1 ELSE 0 END) AS missing_delivery_times_cancelled_orders
    FROM orders
""", con)

display(q3)

# 4. Calculate grand total revenue
print("SQL grand total INR:", round(q2.revenue.sum(), 2))

,name
0,customers
1,orders
2,products


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


,total_orders,completed_orders,cancelled_orders,pending_orders,missing_order_amounts,missing_delivery_times_completed_orders,missing_delivery_times_cancelled_orders
0,12309,11110,1199,0,0,0,1199


SQL grand total INR: 2998668.8


In [62]:
import sqlite3

db_path = ROOT / "data" / "instamart.db"

with sqlite3.connect(db_path) as con:
    Query1 = pd.read_sql_query("""
        SELECT
            count(*) AS high_value_customers
        FROM  (SELECT c.customer_id
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_amount > 500 AND o.status = 'completed'
        GROUP BY c.customer_id
        HAVING SUM(o.order_amount) > 8000
    ) AS high_value_customers
""", con)
    display(Query1)
    Query2 = pd.read_sql_query("""
            SELECT
                c.city,
                p.category,
                ROUND(SUM(o.order_amount), 2) AS revenue
            FROM orders o
            JOIN customers c USING(customer_id)
            JOIN products p USING(product_id)
            WHERE o.status = 'completed'
            GROUP BY c.city, p.category
            ORDER BY c.city, p.category
        """, con)
    
    display(Query2)
    print("SQL grand total INR:", round(Query2.revenue.sum(), 2))
    Query3 = pd.read_sql_query("""
                SELECT
                    count(product_id) AS products_above_average_price
                from products p
                where p.price_inr > (select avg(price_inr) as average from products p )
                """, con)
        
    display(Query3)

    Query4 = pd.read_sql_query("""
                    SELECT COUNT(*) AS customers_without_completed_orders
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
    AND o.status = 'completed'
WHERE o.customer_id IS NULL;               
                    """, con)
            
    display(Query4)

    Query5 = pd.read_sql_query("""
                       WITH customer_total_spending AS (
                         SELECT
        c.customer_id,
        SUM(o.order_amount) AS total_spent
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id
),
customer_average_spending AS (
    SELECT
        AVG(total_spent) AS average_spent
    FROM customer_total_spending
)
SELECT
    ROUND(a.average_spent, 2) AS average_spending,
    COUNT(*) AS customers_above_average_spending
FROM customer_total_spending c
CROSS JOIN customer_average_spending a
WHERE c.total_spent > a.average_spent
GROUP BY a.average_spent;   
                        """, con)
    display(Query5)
    Query6 = pd.read_sql_query("""
    WITH customer_total_spending AS (
    SELECT
        c.customer_id,
        c.city,
        SUM(o.order_amount) AS total_spend
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id, c.city
),
customer_rankings AS (
    SELECT
        customer_id,
        city,
        total_spend,

        RANK() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS spending_rank,

        DENSE_RANK() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS dense_spending_rank,

        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS row_number_rank

    FROM customer_total_spending
)
SELECT *
FROM customer_rankings where spending_rank <> row_number_rank
ORDER BY city, spending_rank, customer_id ;
""", con)
    
    display(Query6)

    Query7 = pd.read_sql_query("""
        WITH customer_total_spending AS (
        SELECT
            c.customer_id,
            c.city,
            SUM(o.order_amount) AS total_spend
        FROM customers c
        JOIN orders o
            ON c.customer_id = o.customer_id
        WHERE o.status = 'completed'
        GROUP BY c.customer_id, c.city
    ),
    customer_rankings AS (
        SELECT
            customer_id,
            city,
            total_spend,
    
            RANK() OVER (
                PARTITION BY city
                ORDER BY total_spend DESC
            ) AS spending_rank,
    
            DENSE_RANK() OVER (
                PARTITION BY city
                ORDER BY total_spend DESC
            ) AS dense_spending_rank,
    
            ROW_NUMBER() OVER (
                PARTITION BY city
                ORDER BY total_spend DESC
            ) AS row_number_rank
    
        FROM customer_total_spending
    )
    SELECT
    customer_id,
    city,
    total_spend,
    spending_rank
FROM customer_rankings
WHERE spending_rank = 1
ORDER BY city;
    """, con)
        
    display(Query7)
    Query8 = pd.read_sql_query("""
    WITH daily_revenue AS (
    SELECT
        SUBSTR(order_ts, 1, 10) AS day,
        SUM(order_amount) AS revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY SUBSTR(order_ts, 1, 10)
)
SELECT
    day,
    ROUND(revenue, 2) AS revenue,
    ROUND(
        SUM(revenue) OVER (
            ORDER BY day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2
    ) AS cumulative_revenue
FROM daily_revenue
ORDER BY day;
""", con)
        
    display(Query8)

,high_value_customers
0,30


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


SQL grand total INR: 2998668.8


,products_above_average_price
0,11


,customers_without_completed_orders
0,25


,average_spending,customers_above_average_spending
0,2552.06,550


,customer_id,city,total_spend,spending_rank,dense_spending_rank,row_number_rank
0,C0734,Bengaluru,2598.00,99,99,100
1,C0723,Bengaluru,135.00,179,178,180
2,C1053,Mumbai,396.00,149,149,150
3,C0608,Mumbai,280.00,160,159,161
4,C1064,Pune,3236.95,74,74,75
5,C0057,Pune,285.00,189,188,190
6,C0936,Pune,285.00,189,188,191
7,C0276,Pune,114.00,209,206,210


,customer_id,city,total_spend,spending_rank
0,C0082,Bengaluru,12856.20,1
1,C0930,Chennai,10151.40,1
2,C0291,Delhi,13742.60,1
3,C0743,Hyderabad,12730.05,1
4,C0693,Mumbai,14766.55,1
5,C0426,Pune,10377.10,1


,day,revenue,cumulative_revenue
0,2025-04-01,16876.50,16876.50
1,2025-04-02,19854.55,36731.05
2,2025-04-03,28048.35,64779.40
3,2025-04-04,14249.60,79029.00
4,2025-04-05,16031.50,95060.50
...,...,...,...
178,2025-09-26,17190.85,2942763.75
179,2025-09-27,13689.95,2956453.70
180,2025-09-28,12689.85,2969143.55
181,2025-09-29,14022.15,2983165.70


Ties cause RANK() to assign the same rank to multiple customers and skip subsequent rank numbers, whereas ROW_NUMBER() assigns a unique sequential number to every customer.

Pandas cross-check and city-wise analysis

1) Load cleaned datasets and prepare completed orders

In [66]:
import pandas as pd
import numpy as np

CLEAN = ROOT / "data" / "clean"

customers = pd.read_csv(CLEAN / "customers.csv")
products = pd.read_csv(CLEAN / "products.csv")
orders = pd.read_csv(CLEAN / "orders.csv")

# Filter completed orders
completed_orders = orders[
    orders["status"] == "completed"
].copy()

completed_orders["order_amount"] = pd.to_numeric(
    completed_orders["order_amount"],
    errors="coerce"
)


2. Join the three tables
We need customer city and product category alongside each completed order.

In [67]:
analysis_df = (
    completed_orders
    .merge(
        customers[["customer_id", "city"]],
        on="customer_id",
        how="left"
    )
    .merge(
        products[["product_id", "category"]],
        on="product_id",
        how="left"
    )
)

display(analysis_df.head())

,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes,city,category
0,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20.0,Chennai,Grocery
1,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23.0,Chennai,Grocery
2,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23.0,Chennai,Grocery
3,ORD-100004,C0001,P023,2,2025-07-29 22:27:32,completed,20,336.00,33.0,Chennai,Snacks
4,ORD-100005,C0002,P022,2,2025-07-18 15:52:20,completed,0,90.00,17.0,Delhi,Snacks


3. Rebuild SQL Q2 using pd.pivot_table()
The SQL Q2 calculated completed-order revenue by city and product category.

In [68]:
pivot_table = pd.pivot_table(
    analysis_df,
    values="order_amount",
    index="city",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Grand Total"
)

display(pivot_table.round(2))

category,Dairy,Electronics,Grocery,Personal Care,Snacks,Grand Total
city,,,,,,
Bengaluru,87075.05,233909.45,110915.65,20193.85,71549.40,523643.40
Chennai,78301.80,195508.80,93817.45,27675.40,68325.70,463629.15
Delhi,117271.70,158549.10,112658.05,23471.50,90879.45,502829.80
Hyderabad,83731.00,228913.10,109475.40,24800.05,81670.35,528589.90
Mumbai,75209.85,214738.40,78970.15,38018.00,72122.10,479058.50
Pune,90349.55,187629.25,108456.60,34275.15,80207.50,500918.05
Grand Total,531938.95,1219248.10,614293.30,168433.95,464754.50,2998668.80


4. Cross-check pandas grand total against SQL

In [69]:
pandas_total = analysis_df["order_amount"].sum()

# Grand total from the pivot table
pivot_total = pivot_table.loc["Grand Total", "Grand Total"]

# Total from your earlier SQL Q2 result
sql_total = q2["revenue"].sum()

print("Pandas total:", round(pandas_total, 2))
print("Pivot table total:", round(pivot_total, 2))
print("SQL Q2 total:", round(sql_total, 2))

assert round(pandas_total, 2) == round(sql_total, 2)
assert round(pivot_total, 2) == round(sql_total, 2)


Pandas total: 2998668.8
Pivot table total: 2998668.8
SQL Q2 total: 2998668.8


5. City-wise sum, mean and count using named aggregation

In [70]:
city_summary = (
    analysis_df
    .groupby("city")
    .agg(
        total_revenue=("order_amount", "sum"),
        average_order_amount=("order_amount", "mean"),
        order_count=("order_amount", "count")
    )
    .round(2)
    .sort_index()
)

display(city_summary)

,total_revenue,average_order_amount,order_count
city,,,
Bengaluru,523643.40,276.47,1894
Chennai,463629.15,272.72,1700
Delhi,502829.80,243.62,2064
Hyderabad,528589.90,284.95,1855
Mumbai,479058.50,297.00,1613
Pune,500918.05,252.48,1984


Part B — Exploratory Data Analysis (EDA)
6. Calculate mean, median and skewness

In [71]:
amount = completed_orders["order_amount"].dropna()

eda_statistics = pd.DataFrame({
    "Statistic": ["Mean", "Median", "Skewness"],
    "Value": [
        amount.mean(),
        amount.median(),
        amount.skew()
    ]
})

display(eda_statistics.round(2))

,Statistic,Value
0,Mean,269.91
1,Median,135.00
2,Skewness,4.61


7. Calculate quartiles and IQR outlier fences
The Interquartile Range (IQR) measures the spread of the middle 50% of order amounts.

In [72]:
q1 = amount.quantile(0.25)
q2 = amount.quantile(0.50)
q3 = amount.quantile(0.75)

iqr = q3 - q1

lower_fence = q1 - (1.5 * iqr)
upper_fence = q3 + (1.5 * iqr)

outliers_below = (amount < lower_fence).sum()
outliers_above = (amount > upper_fence).sum()

iqr_summary = pd.DataFrame({
    "Metric": [
        "Q1 (25th percentile)",
        "Q2 (Median)",
        "Q3 (75th percentile)",
        "IQR",
        "Lower fence",
        "Upper fence",
        "Outliers below lower fence",
        "Outliers above upper fence"
    ],
    "Value": [
        q1,
        q2,
        q3,
        iqr,
        lower_fence,
        upper_fence,
        outliers_below,
        outliers_above
    ]
})

display(iqr_summary.round(2))

,Metric,Value
0,Q1 (25th percentile),64.8
1,Q2 (Median),135.0
2,Q3 (75th percentile),297.0
3,IQR,232.2
4,Lower fence,-283.5
5,Upper fence,645.3
6,Outliers below lower fence,0.0
7,Outliers above upper fence,897.0


8. Calculate the correlation matrix
We need to study the relationship between order amount, discount percentage, quantity and delivery time.

In [73]:
correlation_columns = [
    "order_amount",
    "discount_pct",
    "quantity",
    "delivery_minutes"
]

# Convert columns to numeric
for col in correlation_columns:
    completed_orders[col] = pd.to_numeric(
        completed_orders[col],
        errors="coerce"
    )

correlation_matrix = completed_orders[
    correlation_columns
].corr()

display(correlation_matrix.round(3))

,order_amount,discount_pct,quantity,delivery_minutes
order_amount,1.000,-0.095,0.087,-0.003
discount_pct,-0.095,1.000,-0.000,-0.010
quantity,0.087,-0.000,1.000,-0.013
delivery_minutes,-0.003,-0.010,-0.013,1.000


Identify the strongest correlation pair

In [74]:
# Get only the upper triangle to avoid duplicate pairs
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

# Find the pair with the largest absolute correlation
correlation_pairs = upper_triangle.stack()

strongest_pair = correlation_pairs.abs().idxmax()
strongest_value = correlation_pairs.loc[strongest_pair]

print("Strongest pair:", strongest_pair)
print("Correlation:", round(strongest_value, 3))
print("Absolute correlation:", round(abs(strongest_value), 3))

Strongest pair: ('order_amount', 'discount_pct')
Correlation: -0.095
Absolute correlation: 0.095


## 5. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The model module scales RFM for K-means, compares K values, uses PCA to inspect compressed variance, and builds a lapse model. The lapse model's input features use only orders before 2025-09-01; September is reserved to define the label. The scaler and encoder are fitted inside a training pipeline, not on the test data.

In [2]:
import os

os.chdir(r"C:\Users\Preetha\Downloads\Swiggy-Instamart-Copilot-main\Swiggy-Instamart-Copilot-main - Karthik\instamart_customer_copilot_teaching_kit")

print(os.getcwd())

C:\Users\Preetha\Downloads\Swiggy-Instamart-Copilot-main\Swiggy-Instamart-Copilot-main - Karthik\instamart_customer_copilot_teaching_kit


In [3]:
import os
import sys

print("src folder exists:", os.path.isdir("src"))
print("customer_models.py exists:", os.path.isfile("src/customer_models.py"))

sys.path.insert(0, os.getcwd())

src folder exists: True
customer_models.py exists: True


In [4]:

import pandas as pd
from src.customer_models import (
    rfm_and_segments,
    build_lapse_model,
    build_rfm_table,
    actual_rfm
)

# =====================================================
# PART 1: RFM TABLE
# =====================================================

print("\n========== PART 1: RFM TABLE ==========")

rfm_table = build_rfm_table()

rfm_actual = actual_rfm(rfm_table)
display(rfm_actual)

print("\nFirst 10 customer RFM records:")
display(rfm_table.head(10))


# =====================================================
# PART 2: CUSTOMER SEGMENTATION
# =====================================================

print("\n========== PART 2: CUSTOMER SEGMENTATION ==========")

rfm, k_diagnostics, segment_profile, pca_variance = (
    rfm_and_segments(k=4)
)

print("\nK-Means diagnostics:")
display(k_diagnostics.round(3))

print("\nCustomer segment profile:")
display(segment_profile.round(2))

print("\nPCA explained variance ratio:", pca_variance)
print("Total explained variance:", pca_variance.sum())

print("\nCustomer segment sample:")
display(rfm.head(10))


# =====================================================
# PART 3: LAPSE MODEL - ALL ASSIGNMENT REQUIREMENTS
# =====================================================

print("\n========== PART 3: LAPSE MODEL ==========")

lapse_summary = build_lapse_model()


# -----------------------------------------------------
# 3A. BASELINE RESULTS
# -----------------------------------------------------

print("\n--- BASELINE RESULTS ---")
display(pd.DataFrame([lapse_summary["baseline"]]))


# -----------------------------------------------------
# 3B. LOGISTIC REGRESSION AND RANDOM FOREST
# -----------------------------------------------------

print("\n--- MODEL TUNING AND TEST RESULTS ---")

model_results = []

for model_name, result in lapse_summary["models"].items():

    model_results.append({
        "Model": model_name,
        "Best Parameters": result["best_params"],
        "Best CV F1": round(result["cv_f1"], 3),
        "Test Precision": round(result["precision"], 3),
        "Test Recall": round(result["recall"], 3),
        "Test F1": round(result["f1"], 3),
        "Confusion Matrix": result["confusion_matrix"]
    })

display(pd.DataFrame(model_results))


# -----------------------------------------------------
# 3C. PREPROCESSING CHECKS
# -----------------------------------------------------

print("\n--- FREQUENCY MEAN COMPARISON ---")

display(pd.DataFrame([
    {
        "Dataset": "Training data",
        "Frequency mean": lapse_summary["frequency_means"]["training"]
    },
    {
        "Dataset": "Full cohort",
        "Frequency mean": lapse_summary["frequency_means"]["full_cohort"]
    }
]))

print("\nThe means may differ because the scaler is fitted only on training rows, whereas the full-cohort mean includes both training and test customers.")


# -----------------------------------------------------
# 3D. RANDOM FOREST FEATURE IMPORTANCE
# -----------------------------------------------------

print("\n--- TOP 5 RANDOM FOREST FEATURES ---")

display(
    lapse_summary["feature_importance"].head(5)
)


# -----------------------------------------------------
# 3E. LOGISTIC REGRESSION COEFFICIENTS
# -----------------------------------------------------

print("\n--- LOGISTIC REGRESSION COEFFICIENTS ---")

display(
    lapse_summary["logistic_coefficients"].head(10)
)


# -----------------------------------------------------
# 3F. CLASS BALANCE EXPERIMENT
# -----------------------------------------------------

print("\n--- CLASS BALANCE EXPERIMENT ---")

balanced_results = lapse_summary["balanced_logistic"]

original_results = lapse_summary["models"]["logistic"]

balance_comparison = pd.DataFrame([
    {
        "Model": "Original Logistic Regression",
        "Precision": original_results["precision"],
        "Recall": original_results["recall"],
        "F1": original_results["f1"]
    },
    {
        "Model": "Balanced Logistic Regression",
        "Precision": balanced_results["precision"],
        "Recall": balanced_results["recall"],
        "F1": balanced_results["f1"]
    }
])

display(balance_comparison.round(3))


# -----------------------------------------------------
# 3G. MODEL PERSISTENCE
# -----------------------------------------------------

print("\n--- MODEL PERSISTENCE ---")

print("Model location: outputs/lapse_model.joblib")
print("Model saved, reloaded and prediction equality checked inside build_lapse_model().")


# -----------------------------------------------------
# 3H. PREMIUM CUSTOMER RETENTION LIST
# -----------------------------------------------------

print("\n--- PREMIUM CUSTOMER RETENTION LIST ---")

display(lapse_summary["retention_list"])

print(
    "Number of Premium customers in retention list:",
    len(lapse_summary["retention_list"])
)

print(
    "Number of customers who actually lapsed:",
    lapse_summary["actual_lapsed_in_retention_list"]
)


# =====================================================
# PART 4: FINAL SUMMARY
# =====================================================

print("\n========== FINAL PROJECT SUMMARY ==========")

print("Total RFM customers:", len(rfm_table))
print("Total lapse-model cohort:", lapse_summary["cohort_rows"])
print("Total lapsed customers:", lapse_summary["lapsed"])

print("\nAssignment execution completed!")



========== PART 1: RFM TABLE ==========

=== RFM SUMMARY ===
           Metric  Actual value
   Customer count       1175.00
   Recency median         19.00
  Recency minimum          0.00
  Recency maximum        180.00
   Frequency mean          9.46
Frequency maximum         33.00
    Monetary mean       2552.06
 Monetary maximum      14766.55


,Metric,Actual value
0,Customer count,1175.000000
1,Recency median,19.000000
2,Recency minimum,0.000000
3,Recency maximum,180.000000
4,Frequency mean,9.455319
5,Frequency maximum,33.000000
6,Monetary mean,2552.058553
7,Monetary maximum,14766.550000



First 10 customer RFM records:


,customer_id,recency,frequency,monetary
0,C0001,14,4,512.85
1,C0002,28,7,4644.90
2,C0003,7,22,3644.85
3,C0004,0,5,369.00
4,C0005,147,2,324.00
5,C0006,34,1,90.00
6,C0007,3,18,2600.00
7,C0008,3,15,3096.50
8,C0009,19,11,8424.80
9,C0010,66,4,450.00



========== PART 2: CUSTOMER SEGMENTATION ==========

K-Means diagnostics:


,k,inertia,silhouette
0,2,1820.761,0.465
1,3,1268.235,0.445
2,4,743.071,0.526
3,5,617.367,0.480
4,6,503.171,0.482
5,7,408.460,0.416
6,8,366.503,0.412


... (15 outputs left out)

,Model,Precision,Recall,F1
0,Original Logistic Regression,0.644,0.652,0.648
1,Balanced Logistic Regression,0.589,0.854,0.697



--- MODEL PERSISTENCE ---
Model location: outputs/lapse_model.joblib
Model saved, reloaded and prediction equality checked inside build_lapse_model().

--- PREMIUM CUSTOMER RETENTION LIST ---


,customer_id,lapse_probability,actually_lapsed
13,C0862,0.510513,0
190,C0267,0.477212,0
165,C1008,0.473260,0
227,C0060,0.471746,0
5,C0802,0.471582,0
216,C0942,0.465245,0
144,C0553,0.462799,0
210,C0021,0.461252,0
82,C0133,0.460743,0
123,C0565,0.458271,1


Number of Premium customers in retention list: 10
Number of customers who actually lapsed: 1

========== FINAL PROJECT SUMMARY ==========
Total RFM customers: 1175
Total lapse-model cohort: 1156
Total lapsed customers: 443

Assignment execution completed!


Inertia measures the sum of squared distances between data points and their assigned cluster centroids. As the number of clusters (K) increases, inertia generally decreases because data points are grouped into smaller, more compact clusters, so K=8 will have lower inertia than K=2. However, choosing K based only on lower inertia is not appropriate because increasing K can always improve compactness, even when it creates unnecessary clusters; therefore, methods like the Elbow Method help identify a suitable K.

In [7]:
import importlib
import src.customer_models as cm

importlib.reload(cm)

lapse_summary = cm.build_lapse_model()


=== MISSING VALUE CHECK: RAW FEATURES ===
recency               0
frequency             0
monetary              0
avg_discount          0
cancellation_rate     0
avg_delivery          0
favourite_category    0
city                  0
age                   0
gender                0
tenure_days           0
dtype: int64

=== BASELINE RESULTS ===
{'accuracy': 0.6163793103448276, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'confusion_matrix': [[143, 0], [89, 0]]}

=== LOGISTIC RESULTS ===
Best parameters: {'model__C': 100}
Best CV F1: 0.651
Confusion matrix:
[[111, 32], [31, 58]]
Precision: 0.644
Recall: 0.652
F1: 0.648

=== FOREST RESULTS ===
Best parameters: {'model__max_depth': 8, 'model__min_samples_leaf': 10}
Best CV F1: 0.687
Confusion matrix:
[[110, 33], [32, 57]]
Precision: 0.633
Recall: 0.64
F1: 0.637

=== FREQUENCY MEAN COMPARISON ===
Training scaler frequency mean: 8.114718614718615
Full-cohort frequency mean: 8.028546712802768

No missing values in transformed training features

In [10]:
for name, result in lapse_summary["models"].items():
    print("\nModel:", name)
    print("Best CV F1:", result["cv_f1"])
    print("Test F1:", result["f1"])


Model: logistic
Best CV F1: 0.650552962900789
Test F1: 0.6480446927374302

Model: forest
Best CV F1: 0.6865517479789224
Test F1: 0.6368715083798883


In [11]:
for name, result in lapse_summary["models"].items():
    if result["f1"] >= 0.55:
        print(name, "MEETS the F1 requirement")
    else:
        print(name, "DOES NOT MEET the F1 requirement")

logistic MEETS the F1 requirement
forest MEETS the F1 requirement


## 6. Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

Reload the module

In [3]:
import importlib
import src.triage as triage

importlib.reload(triage)

Working directory: C:\Users\Preetha\Downloads\Swiggy-Instamart-Copilot-main\Swiggy-Instamart-Copilot-main - Karthik\instamart_customer_copilot_teaching_kit
Working directory: C:\Users\Preetha\Downloads\Swiggy-Instamart-Copilot-main\Swiggy-Instamart-Copilot-main - Karthik\instamart_customer_copilot_teaching_kit


<module 'src.triage' from 'c:\\Users\\Preetha\\Downloads\\Swiggy-Instamart-Copilot-main\\Swiggy-Instamart-Copilot-main - Karthik\\instamart_customer_copilot_teaching_kit\\src\\triage.py'>

Zero-shot experiment

In [4]:
zero_results, zero_summary = triage.run_pipeline(
    prompt=triage.ZERO,
    token_budget=100000
)

print("ZERO-SHOT RESULTS")
print("Classified:", zero_summary["classified_tickets"])
print("Accuracy:", round(zero_summary["accuracy"] * 100, 2), "%")
print("Total tokens:", zero_summary["total_tokens"])
print("Tokens per classified ticket:",
      round(zero_summary["tokens_per_classified_ticket"], 2))

print("\nRedaction counts:")
print(zero_summary["redactions"])
print("Total redactions:", zero_summary["redactions_total"])

print("\nRemaining sensitive information:")
print(zero_summary["remaining_sensitive_matches"])

print("\nStatus counts:")
print(zero_summary["status_counts"])

print("\nZero-shot confusion matrix:")
display(zero_summary["confusion_matrix"])

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
ZERO-SHOT RESULTS
Classified: 147
Accuracy: 85.71 %
Total tokens: 9712
Tokens per classified ticket: 66.07

Redaction counts:
{'email': 55, 'phone': 91, 'order_id': 132}
Total redactions: 278

Remaining sensitive information:
{'email': 0, '

category,billing,delivery,fraud,technical
true_category,,,,
billing,38,0,0,0
delivery,0,36,0,0
fraud,11,0,23,0
technical,10,0,0,29


Few-shot experiment

In [5]:
few_results, few_summary = triage.run_pipeline(
    prompt=triage.FEW,
    token_budget=100000
)

print("FEW-SHOT RESULTS")
print("Classified:", few_summary["classified_tickets"])
print("Accuracy:", round(few_summary["accuracy"] * 100, 2), "%")
print("Total tokens:", few_summary["total_tokens"])
print("Tokens per classified ticket:",
      round(few_summary["tokens_per_classified_ticket"], 2))

print("\nStatus counts:")
print(few_summary["status_counts"])

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
FEW-SHOT RESULTS
Classified: 147
Accuracy: 100.0 %
Total tokens: 17512
Tokens per classified ticket: 119.13

Status counts:
{'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9, 'skipped_empty': 4, 'flagged_long': 

Compare both prompts

In [7]:
import pandas as pd
comparison = pd.DataFrame([
    {
        "Prompt": "Zero-shot",
        "Classified": zero_summary["classified_tickets"],
        "Accuracy (%)": round(zero_summary["accuracy"] * 100, 2),
        "Total tokens": zero_summary["total_tokens"],
        "Tokens per ticket": round(
            zero_summary["tokens_per_classified_ticket"], 2
        )
    },
    {
        "Prompt": "Few-shot",
        "Classified": few_summary["classified_tickets"],
        "Accuracy (%)": round(few_summary["accuracy"] * 100, 2),
        "Total tokens": few_summary["total_tokens"],
        "Tokens per ticket": round(
            few_summary["tokens_per_classified_ticket"], 2
        )
    }
])

display(comparison)

,Prompt,Classified,Accuracy (%),Total tokens,Tokens per ticket
0,Zero-shot,147,85.71,9712,66.07
1,Few-shot,147,100.00,17512,119.13


Zero-shot budget demonstration

In [8]:
budget_results, budget_summary = triage.run_pipeline(
    prompt=triage.ZERO,
    token_budget=6000,
    output_file="triage_budget_demo.csv"
)

print("Budget used: 6000")
print("Total tokens consumed:", budget_summary["total_tokens"])

print(
    "Budget exhausted tickets:",
    budget_summary["status_counts"].get("budget_exhausted", 0)
)

display(budget_results.head(10))

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
Budget used: 6000
Total tokens consumed: 6049
Budget exhausted tickets: 59


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At Risk,HIGH,security_team,66
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,62
2,T0095,C1087,classified_json,billing,At Risk,NORMAL,support_queue,64
3,T0159,C1172,skipped_empty,,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,billing,At Risk,NORMAL,support_queue,66
5,T0124,C0139,classified_json,billing,Frequent,NORMAL,support_queue,66
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,58
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,61
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,62
9,T0002,C0881,classified_json,delivery,At Risk,NORMAL,support_queue,60


Final few-shot pipeline

In [9]:
final_results, final_summary = triage.run_pipeline(
    prompt=triage.FEW,
    token_budget=100000,
    output_file="triage_results.csv"
)

print("FINAL TRIAGE RESULTS")
print(final_summary["status_counts"])

print("\nTotal tokens:", final_summary["total_tokens"])
print("Accuracy:", round(final_summary["accuracy"] * 100, 2), "%")

display(final_results.head(10))

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
FINAL TRIAGE RESULTS
{'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9, 'skipped_empty': 4, 'flagged_long': 4}

Total tokens: 17512
Accuracy: 100.0 %


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At Risk,HIGH,security_team,116
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,112
2,T0095,C1087,classified_json,technical,At Risk,NORMAL,support_queue,114
3,T0159,C1172,skipped_empty,,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,At Risk,HIGH,security_team,116
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,116
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,108
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,111
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,112
9,T0002,C0881,classified_json,delivery,At Risk,NORMAL,support_queue,110


In [10]:
import pandas as pd
from pathlib import Path

output_dir = Path("outputs")

budget_df = pd.read_csv(output_dir / "triage_budget_demo.csv")
final_df = pd.read_csv(output_dir / "triage_results.csv")

print("Budget demo rows:", len(budget_df))
print(
    "Budget exhausted:",
    (budget_df["status"] == "budget_exhausted").sum()
)

print("\nFinal output rows:", len(final_df))
print("Final columns:", final_df.columns.tolist())
print("\nFinal routing counts:")
print(final_df["route"].value_counts())

assert len(final_df) == 164

assert final_df.columns.tolist() == [
    "ticket_id", "customer_id", "status", "category",
    "segment_name", "priority", "route", "tokens_used"
]

assert (budget_df["status"] == "budget_exhausted").sum() > 0

print("\nCSV acceptance checks passed.")

Budget demo rows: 164
Budget exhausted: 59

Final output rows: 164
Final columns: ['ticket_id', 'customer_id', 'status', 'category', 'segment_name', 'priority', 'route', 'tokens_used']

Final routing counts:
route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64

CSV acceptance checks passed.


## 7. Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.

In [15]:
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

Policy chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."


{'answer': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.', 'sources': ['refund_policy.txt', 'privacy_policy.txt'], 'retrieved': [{'source': 'refund_policy.txt', 'score': 0.35464416504570484, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.31425507733397284, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'privacy_policy.txt', 'score': 0.15759666156595936, 'text': 'Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account deletion and the request is c

,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,True,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,True,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False


Expected-document hits: 12
Expected-fact hits: 12
Fallbacks: 2


## 8. Part D — refund tools and safe stopping

The refund agent demonstrates a tool loop: look up an order in SQLite, retrieve policy, calculate the eligible refund, then issue it or escalate to a human. In this teaching kit the ledger is a local JSONL file; no money moves.

Before running agent demonstrations, inspect `src/refund_agent.py` and ask students to trace how a tool name maps to a Python function.

In [16]:
from src.refund_agent import get_order, search_policy, calculate_refund, TOOL_MAP, run_agent
print("Tools:", sorted(TOOL_MAP))
print("Example order lookup:")
with sqlite3.connect(db_path) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print(example_order, get_order(example_order))
print("Refund calculation:", calculate_refund(250.0, 52))
print("Timeout safety demo:", run_agent(f"Refund order {example_order}", scenario_id="timeout-demo", time_limit_seconds=0))

Tools: ['calculate_refund', 'escalate_to_human', 'get_order', 'issue_refund', 'search_policy']
Example order lookup:
ORD-100048 {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65}
Refund calculation: {'refund_inr': 250.0}
Timeout safety demo: {'status': 'FALLBACK_TIMEOUT', 'final_answer': 'FALLBACK_TIMEOUT: hand off to a human agent.', 'steps': [], 'llm_calls': 0, 'tokens_used': 0}


## 9. Final generated artifacts and reproducibility check

Review the CSVs and model file generated by the pipeline. In class, have students delete `data/clean/`, `data/instamart.db`, and `outputs/`, then rerun the documented commands to verify that the project can be rebuilt from source.


In [17]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())

\ndata\clean
['customers.csv', 'orders.csv', 'products.csv']
\noutputs
['customer_segments.csv', 'lapse_model.joblib', 'retention_list.csv', 'triage_results.csv']
\nstate
[]
Database exists: True
